# 02 · Integridad relacional y matriz OD — Bogotá 2023

**Objetivo.** Validar las claves confirmadas de la Encuesta de Movilidad 2023 sobre las tablas completas y construir matrices origen–destino ponderadas por ZAT.

Esta es la **Compuerta 3**. Produce insumos descriptivos reproducibles; todavía no estima tiempos de primer paso ni accesibilidad.


In [ ]:
%pip -q install pandas geopandas pyarrow requests


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import csv
import hashlib
import json
import re
import time
import unicodedata
import zipfile

import geopandas as gpd
import numpy as np
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

ROOT = Path("/content/colombia-stochastic-access")
RAW = ROOT / "data/raw/bogota_em2023"
INTERIM = ROOT / "data/interim/bogota_em2023_gate3"
OUT = ROOT / "outputs/bogota_em2023/gate3"
for directory in (RAW, INTERIM, OUT):
    directory.mkdir(parents=True, exist_ok=True)

SOURCES = {
    "processed_edoh": "https://observatorio.movilidadbogota.gov.co/sites/default/files/2025-09/05_Base%20datos%20procesada%20EODH.zip",
    "zoning_edoh": "https://observatorio.movilidadbogota.gov.co/sites/default/files/2025-09/03_Zonificacion%20EODH%20%281%29.zip",
}

print("Workspace:", ROOT)


## 1. Descarga y extracción reproducible

Se reutilizan los ZIP descargados por el notebook 01 cuando están disponibles. Los archivos grandes permanecen fuera de Git.


In [ ]:
def sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def robust_session():
    retry = Retry(
        total=3,
        connect=3,
        read=3,
        status=3,
        backoff_factor=0.8,
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=frozenset(["GET"]),
        raise_on_status=False,
    )
    session = requests.Session()
    session.headers.update({
        "User-Agent": "Mozilla/5.0 ColombiaStochasticAccess/1.0",
        "Accept": "*/*",
        "Connection": "keep-alive",
    })
    session.mount("https://", HTTPAdapter(max_retries=retry))
    session.mount("http://", HTTPAdapter(max_retries=retry))
    return session


def download(name, url, timeout=(30, 180), max_attempts=5):
    destination = RAW / f"{name}.zip"
    destination.parent.mkdir(parents=True, exist_ok=True)

    if destination.exists() and zipfile.is_zipfile(destination):
        retrieved = "cached"
    else:
        if destination.exists():
            corrupt = destination.with_suffix(
                destination.suffix + f".invalid-{int(time.time())}"
            )
            destination.replace(corrupt)
            print(f"ZIP previo inválido conservado como: {corrupt.name}")

        temporary = destination.with_suffix(destination.suffix + ".part")

        with robust_session() as session:
            for attempt in range(1, max_attempts + 1):
                offset = temporary.stat().st_size if temporary.exists() else 0
                headers = {"Range": f"bytes={offset}-"} if offset else {}

                try:
                    with session.get(
                        url,
                        stream=True,
                        timeout=timeout,
                        headers=headers,
                        allow_redirects=True,
                    ) as response:
                        if response.status_code == 416 and temporary.exists():
                            if zipfile.is_zipfile(temporary):
                                temporary.replace(destination)
                                break
                            temporary.unlink()
                            raise OSError(
                                "El parcial no es un ZIP completo; se reiniciará."
                            )

                        response.raise_for_status()
                        resumed = offset > 0 and response.status_code == 206
                        mode = "ab" if resumed else "wb"
                        completed = offset if resumed else 0

                        content_range = response.headers.get("content-range", "")
                        if "/" in content_range:
                            total = int(content_range.rsplit("/", 1)[1])
                        else:
                            response_bytes = int(
                                response.headers.get("content-length", 0)
                            )
                            total = (
                                completed + response_bytes
                                if response_bytes
                                else 0
                            )

                        with temporary.open(mode) as handle:
                            for chunk in response.iter_content(1024 * 1024):
                                if chunk:
                                    handle.write(chunk)
                                    completed += len(chunk)
                                    if total:
                                        print(
                                            f"\r{destination.name}: "
                                            f"{100 * completed / total:5.1f}%",
                                            end="",
                                        )

                    if total and temporary.stat().st_size < total:
                        raise OSError(
                            f"Descarga incompleta: "
                            f"{temporary.stat().st_size} de {total} bytes"
                        )
                    if not zipfile.is_zipfile(temporary):
                        raise OSError("La respuesta descargada no es un ZIP válido.")

                    temporary.replace(destination)
                    print()
                    break

                except (requests.RequestException, OSError) as exc:
                    if attempt == max_attempts:
                        raise ConnectionError(
                            f"No se pudo descargar {url} tras "
                            f"{max_attempts} intentos. El archivo .part se "
                            "conserva para reanudar al ejecutar de nuevo."
                        ) from exc
                    wait_seconds = min(2 ** (attempt - 1), 30)
                    print(
                        f"\nIntento {attempt}/{max_attempts} interrumpido: "
                        f"{exc}. Reintentando en {wait_seconds} s..."
                    )
                    time.sleep(wait_seconds)

        retrieved = datetime.now(timezone.utc).isoformat()

    return {
        "name": name,
        "url": url,
        "path": str(destination),
        "bytes": int(destination.stat().st_size),
        "sha256": sha256(destination),
        "retrieved_utc": retrieved,
    }


def extract_recursive(zip_path, destination):
    destination.mkdir(parents=True, exist_ok=True)
    queue = [(zip_path, destination)]
    visited = set()
    while queue:
        current_zip, current_destination = queue.pop(0)
        signature = (str(current_zip.resolve()), str(current_destination.resolve()))
        if signature in visited:
            continue
        visited.add(signature)
        with zipfile.ZipFile(current_zip) as archive:
            archive.extractall(current_destination)
        for nested in current_destination.rglob("*.zip"):
            nested_destination = nested.parent / nested.stem
            nested_signature = (
                str(nested.resolve()),
                str(nested_destination.resolve()),
            )
            if nested_signature not in visited:
                queue.append((nested, nested_destination))


manifest = pd.DataFrame(
    [download(name, url) for name, url in SOURCES.items()]
)
for row in manifest.itertuples(index=False):
    extract_recursive(Path(row.path), INTERIM / row.name)

manifest.to_csv(OUT / "source_manifest.csv", index=False)
display(manifest)


## 2. Lectura selectiva de las tablas completas

Los identificadores se conservan como texto. Solo se cargan las columnas necesarias para las pruebas relacionales y la matriz OD, lo que reduce memoria y evita alterar ceros iniciales.


In [ ]:
def normalize(value):
    value = unicodedata.normalize("NFKD", str(value))
    value = "".join(char for char in value if not unicodedata.combining(char))
    value = value.lower().strip()
    value = re.sub(r"[^a-z0-9]+", "_", value)
    return value.strip("_")


def csv_format(path):
    for encoding in ("utf-8-sig", "latin-1"):
        try:
            with path.open("r", encoding=encoding, errors="strict") as handle:
                sample = handle.read(32768)
            delimiter = csv.Sniffer().sniff(sample, delimiters=",;\t|").delimiter
            return encoding, delimiter
        except (UnicodeDecodeError, csv.Error):
            continue
    raise ValueError(f"No fue posible detectar formato CSV: {path}")


def read_selected(path, desired):
    encoding, delimiter = csv_format(path)
    header = pd.read_csv(path, sep=delimiter, encoding=encoding, nrows=0)
    mapping = {normalize(column): column for column in header.columns}
    missing = sorted(set(desired) - set(mapping))
    if missing:
        print(f"Aviso {path.name}: columnas opcionales ausentes: {missing}")
    selected = [mapping[column] for column in desired if column in mapping]
    frame = pd.read_csv(
        path,
        sep=delimiter,
        encoding=encoding,
        usecols=selected,
        dtype="string",
        low_memory=False,
    )
    frame.columns = [normalize(column) for column in frame.columns]
    return frame


def select_csv(role):
    paths = list((INTERIM / "processed_edoh").rglob("*.csv"))
    matches = []
    for path in paths:
        label = normalize(path.stem)
        if role == "households" and re.search(r"hogar", label):
            matches.append(path)
        elif role == "persons" and re.search(r"persona", label):
            matches.append(path)
        elif role == "trips" and re.search(r"viaje", label) and not re.search(r"etapa", label):
            matches.append(path)
        elif role == "stages" and re.search(r"etapa", label):
            matches.append(path)
    if not matches:
        raise FileNotFoundError(f"No se encontró CSV para {role}")
    return sorted(matches, key=lambda path: (len(path.parts), len(path.name)))[0]


files = {role: select_csv(role) for role in ("households", "persons", "trips", "stages")}
for role, path in files.items():
    print(f"{role:12s} -> {path.relative_to(INTERIM)}")

households = read_selected(files["households"], ["key_hg"])
persons = read_selected(files["persons"], ["key_hg", "key_persona", "cod_per"])
trips = read_selected(
    files["trips"],
    [
        "key_hg", "key_pers", "key_pers_viaja", "cod_pers", "key_viaje",
        "fexp_vj", "zat_ori", "zat_des", "duracion_min",
        "nom_mun_ori", "nom_mun_des",
    ],
)
stages = read_selected(files["stages"], ["key_viaje", "key_etapa"])

table_sizes = pd.Series({
    "households": len(households),
    "persons": len(persons),
    "trips": len(trips),
    "stages": len(stages),
}, name="rows")
display(table_sizes)


## 3. Normalización y pruebas de claves

Se prueban explícitamente las relaciones publicadas. Para persona–viaje se selecciona la pareja con mayor cobertura observada entre los candidatos equivalentes.


In [ ]:
def clean_key(series):
    cleaned = series.astype("string").str.strip()
    cleaned = cleaned.str.replace(r"\.0$", "", regex=True)
    return cleaned.mask(cleaned.isin(["", "nan", "None", "<NA>"]))


def coverage(child, parent):
    child_clean = clean_key(child).dropna()
    parent_set = set(clean_key(parent).dropna().unique())
    if len(child_clean) == 0:
        return 0.0
    return float(child_clean.isin(parent_set).mean())


def uniqueness(series):
    clean = clean_key(series).dropna()
    if len(clean) == 0:
        return 0.0
    return float(clean.nunique() / len(clean))


pair_candidates = [
    ("key_pers", "key_persona"),
    ("key_pers_viaja", "key_persona"),
    ("cod_pers", "cod_per"),
]
pair_scores = []
for trip_key, person_key_candidate in pair_candidates:
    if trip_key in trips and person_key_candidate in persons:
        pair_scores.append({
            "trip_key": trip_key,
            "person_key": person_key_candidate,
            "coverage": coverage(trips[trip_key], persons[person_key_candidate]),
        })

pair_scores = pd.DataFrame(pair_scores).sort_values("coverage", ascending=False)
if pair_scores.empty:
    raise RuntimeError("No hay pareja candidata persona–viaje")
best_pair = pair_scores.iloc[0]
trip_person_key = str(best_pair.trip_key)
person_key = str(best_pair.person_key)

relational_metrics = {
    "household_pk_uniqueness": uniqueness(households["key_hg"]),
    "person_pk_uniqueness": uniqueness(persons["key_persona"]),
    "trip_pk_uniqueness": uniqueness(trips["key_viaje"]),
    "stage_pk_uniqueness": uniqueness(stages["key_etapa"]),
    "person_to_household_coverage": coverage(persons["key_hg"], households["key_hg"]),
    "trip_to_household_coverage": coverage(trips["key_hg"], households["key_hg"]),
    "trip_to_person_coverage": coverage(trips[trip_person_key], persons[person_key]),
    "stage_to_trip_coverage": coverage(stages["key_viaje"], trips["key_viaje"]),
}
display(pair_scores)
display(pd.Series(relational_metrics, name="value"))
print("Pareja persona–viaje elegida:", person_key, "<-", trip_person_key)


## 4. Cobertura de ZAT, ponderadores y duración

La cobertura se mide contra la capa oficial ZAT 2023. Los códigos no cartografiados quedan cuantificados, no se eliminan silenciosamente.


In [ ]:
def parse_number(series):
    raw = series.astype("string").str.strip()
    comma_share = float(raw.str.contains(",", regex=False, na=False).mean())
    if comma_share > 0.05:
        raw = raw.str.replace(".", "", regex=False).str.replace(",", ".", regex=False)
    return pd.to_numeric(raw, errors="coerce")


zat_paths = [
    path for path in (INTERIM / "zoning_edoh").rglob("*.shp")
    if "zat" in normalize(path.stem)
]
if not zat_paths:
    raise FileNotFoundError("No se encontró la capa ZAT")
zat_path = sorted(zat_paths, key=lambda path: len(path.parts))[0]
zat_layer = gpd.read_file(zat_path)
zat_columns = {normalize(column): column for column in zat_layer.columns}
if "zat" not in zat_columns:
    raise KeyError(f"La capa no contiene columna ZAT: {list(zat_layer.columns)}")

zat_official = set(clean_key(zat_layer[zat_columns["zat"]]).dropna().unique())
origin = clean_key(trips["zat_ori"])
destination = clean_key(trips["zat_des"])
weights = parse_number(trips["fexp_vj"])
duration = parse_number(trips["duracion_min"])

def spatial_coverage(series):
    valid = series.dropna()
    return float(valid.isin(zat_official).mean()) if len(valid) else 0.0

quality_metrics = {
    "origin_zat_coverage": spatial_coverage(origin),
    "destination_zat_coverage": spatial_coverage(destination),
    "positive_weight_share": float((weights.dropna() > 0).mean()) if weights.notna().any() else 0.0,
    "valid_duration_share": float((duration.dropna() >= 0).mean()) if duration.notna().any() else 0.0,
    "weight_nonmissing_share": float(weights.notna().mean()),
    "duration_nonmissing_share": float(duration.notna().mean()),
}
display(pd.Series(quality_metrics, name="value"))

unmapped = pd.concat([
    origin[origin.notna() & ~origin.isin(zat_official)].value_counts().rename("origin_rows"),
    destination[destination.notna() & ~destination.isin(zat_official)].value_counts().rename("destination_rows"),
], axis=1).fillna(0).astype(int)
unmapped.to_csv(OUT / "unmapped_zat_codes.csv", index_label="zat")
display(unmapped.head(20))


## 5. Matrices origen–destino ponderadas

Se generan dos matrices largas: toda el área de estudio y viajes internos de Bogotá. La variable weighted_trips usa fexp_vj; mean_duration_min es una media ponderada válida cuando existe duración.


In [ ]:
od_rows = pd.DataFrame({
    "zat_origin": origin,
    "zat_destination": destination,
    "weight": weights,
    "duration_min": duration,
    "municipality_origin": trips["nom_mun_ori"].astype("string"),
    "municipality_destination": trips["nom_mun_des"].astype("string"),
})
od_rows = od_rows[
    od_rows["zat_origin"].notna()
    & od_rows["zat_destination"].notna()
    & od_rows["weight"].gt(0)
].copy()
od_rows["weighted_minutes"] = od_rows["weight"] * od_rows["duration_min"]


def build_od(frame):
    valid_duration = frame["duration_min"].notna()
    duration_weights = frame["weight"].where(valid_duration, 0.0)
    working = frame.assign(duration_weight=duration_weights)
    grouped = (
        working.groupby(["zat_origin", "zat_destination"], observed=True, dropna=False)
        .agg(
            sampled_trips=("weight", "size"),
            weighted_trips=("weight", "sum"),
            weighted_minutes=("weighted_minutes", "sum"),
            duration_weight=("duration_weight", "sum"),
        )
        .reset_index()
    )
    grouped["mean_duration_min"] = (
        grouped["weighted_minutes"] / grouped["duration_weight"].replace(0, np.nan)
    )
    return grouped.drop(columns=["weighted_minutes", "duration_weight"])


od_all = build_od(od_rows)
bogota_origin = od_rows["municipality_origin"].map(normalize).str.contains("bogota", na=False)
bogota_destination = od_rows["municipality_destination"].map(normalize).str.contains("bogota", na=False)
od_bogota = build_od(od_rows[bogota_origin & bogota_destination])

od_all.to_csv(OUT / "od_zat_weighted_all.csv", index=False)
od_bogota.to_csv(OUT / "od_zat_weighted_bogota_internal.csv", index=False)
od_all.to_parquet(OUT / "od_zat_weighted_all.parquet", index=False)
od_bogota.to_parquet(OUT / "od_zat_weighted_bogota_internal.parquet", index=False)

od_summary = pd.DataFrame({
    "scope": ["all", "bogota_internal"],
    "sampled_trips": [int(od_all.sampled_trips.sum()), int(od_bogota.sampled_trips.sum())],
    "weighted_trips": [float(od_all.weighted_trips.sum()), float(od_bogota.weighted_trips.sum())],
    "od_pairs": [len(od_all), len(od_bogota)],
})
od_summary.to_csv(OUT / "od_summary.csv", index=False)
display(od_summary)
display(od_bogota.sort_values("weighted_trips", ascending=False).head(15))


## 6. Compuerta 3

Los umbrales son conservadores y quedan registrados. Un fallo no se corrige eliminando registros: primero se inspeccionan relational_metrics.csv y unmapped_zat_codes.csv.


In [ ]:
thresholds = {
    "pk_uniqueness_min": 0.999,
    "relation_coverage_min": 0.98,
    "zone_coverage_min": 0.95,
    "positive_weight_share_min": 0.99,
    "weight_nonmissing_share_min": 0.95,
    "duration_nonmissing_share_min": 0.90,
}

gate3 = {
    "household_pk_valid": relational_metrics["household_pk_uniqueness"] >= thresholds["pk_uniqueness_min"],
    "person_pk_valid": relational_metrics["person_pk_uniqueness"] >= thresholds["pk_uniqueness_min"],
    "trip_pk_valid": relational_metrics["trip_pk_uniqueness"] >= thresholds["pk_uniqueness_min"],
    "stage_pk_valid": relational_metrics["stage_pk_uniqueness"] >= thresholds["pk_uniqueness_min"],
    "person_household_join_valid": relational_metrics["person_to_household_coverage"] >= thresholds["relation_coverage_min"],
    "trip_household_join_valid": relational_metrics["trip_to_household_coverage"] >= thresholds["relation_coverage_min"],
    "trip_person_join_valid": relational_metrics["trip_to_person_coverage"] >= thresholds["relation_coverage_min"],
    "stage_trip_join_valid": relational_metrics["stage_to_trip_coverage"] >= thresholds["relation_coverage_min"],
    "origin_zone_coverage_valid": quality_metrics["origin_zat_coverage"] >= thresholds["zone_coverage_min"],
    "destination_zone_coverage_valid": quality_metrics["destination_zat_coverage"] >= thresholds["zone_coverage_min"],
    "weights_valid": (
        quality_metrics["positive_weight_share"] >= thresholds["positive_weight_share_min"]
        and quality_metrics["weight_nonmissing_share"] >= thresholds["weight_nonmissing_share_min"]
    ),
    "duration_coverage_valid": quality_metrics["duration_nonmissing_share"] >= thresholds["duration_nonmissing_share_min"],
    "od_matrix_nonempty": len(od_all) > 0 and float(od_all["weighted_trips"].sum()) > 0,
}
gate3 = {key: bool(value) for key, value in gate3.items()}
gate3["gate_3_passed"] = bool(all(gate3.values()))

relational_df = pd.Series(relational_metrics, name="value").rename_axis("metric").reset_index()
quality_df = pd.Series(quality_metrics, name="value").rename_axis("metric").reset_index()
relational_df.to_csv(OUT / "relational_metrics.csv", index=False)
quality_df.to_csv(OUT / "quality_metrics.csv", index=False)

report = {
    "generated_utc": datetime.now(timezone.utc).isoformat(),
    "selected_person_trip_mapping": {
        "person_key": person_key,
        "trip_foreign_key": trip_person_key,
        "coverage": float(best_pair.coverage),
    },
    "thresholds": {key: float(value) for key, value in thresholds.items()},
    "relational_metrics": {key: float(value) for key, value in relational_metrics.items()},
    "quality_metrics": {key: float(value) for key, value in quality_metrics.items()},
    "gate": gate3,
}
(OUT / "gate_3.json").write_text(
    json.dumps(report, ensure_ascii=False, indent=2, allow_nan=False),
    encoding="utf-8",
)

display(pd.Series(gate3, name="result"))
print("Resultados:", OUT)


## Interpretación y siguiente decisión

- Si gate_3_passed=True, el siguiente notebook estimará distribuciones empíricas de tiempo por relación OD/modo/franja y definirá la red de transición.
- Si falla únicamente la cobertura ZAT, revise unmapped_zat_codes.csv: pueden ser zonas externas válidas y deberán documentarse como tales.
- Si falla una clave o cardinalidad, conserve los registros y reporte el patrón antes de decidir una regla.
- Versione los JSON/CSV pequeños. No suba ZIP, tablas extraídas ni matrices OD grandes si exceden el límite razonable del repositorio.
